# **Army CSD Canteen Stock Demand Prediction - Comparative Training**

This notebook implements a complete time series forecasting comparative analysis to predict monthly demand for **3,200+ grocery and liquor items** in the CSD canteen.

### **Objective**
1. **Clean & Parse Data**: Load and clean `data/master_training_data.csv` (DD-MM-YYYY format, clean tail NaN rows).
2. **Train till 2025 Only**: Filter the dataset to exclude 2026 data completely.
3. **Data Completeness & Imputation**: Complete the grid for all items across all 24 months of 2024-2025. Impute missing sales volumes (`Net_Qty`) using linear interpolation (vectorized per product series, keeping pre-introduction sales at 0.0).
4. **Model Comparison**: Compare **ARIMA**, **SARIMA**, and **XGBoost** models on the highest-volume products and compute error metrics (MAE, RMSE, MAPE).
5. **Production Model Export**: Fit the selected best-performing model (XGBoost) on the full dataset and save to the backend target: `model/xgboost_demand_model.json`.

## **1. Setup & Imports**

Load standard Python libraries and statsmodels modules for ARIMA / SARIMA modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import json
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Plotting configuration
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 12

print("Libraries imported successfully!")

## **2. Load and Clean Raw Data**

Load raw sales dataset from the data folder, drop empty rows at the bottom, parse the dates correctly, and filter the dataset to 2024-2025 only.

In [ ]:
# Path to the data file
data_path = Path("data/master_training_data.csv")

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at {data_path}. Please run this from the repository root.")

# Read the CSV
df_raw = pd.read_csv(data_path)

# 1. Clean completely empty rows
df_raw = df_raw.dropna(subset=['Date', 'Item_ID']).reset_index(drop=True)

# 2. Parse Date correctly in DD-MM-YYYY format
df_raw['Date'] = pd.to_datetime(df_raw['Date'], dayfirst=True)

# 3. Filter to train till 2025 only (exclude 2026 data completely)
df_raw = df_raw[df_raw['Date'].dt.year <= 2025].reset_index(drop=True)

print(f"Loaded {len(df_raw)} valid records for 2024-2025.")
print(f"Date range: {df_raw['Date'].min().strftime('%Y-%m-%d')} to {df_raw['Date'].max().strftime('%Y-%m-%d')}")
print(f"Unique Items: {df_raw['Item_Name'].nunique()}")
df_raw.head()

## **3. Data Completeness & Imputation**

We create a full monthly grid from January 2024 to December 2025 for all unique items. For missing sales data, we perform linear interpolation and ensure that dates before the item's first recorded sale are set to 0.0 sales. Lags and rolling metrics are re-calculated on the completed timeline.

In [ ]:
print("Generating complete grid of all items and all 24 months...")
# Create a complete monthly grid from 2024-01-01 to 2025-12-01
all_items = df_raw['Item_ID'].unique()
all_months = pd.date_range(start="2024-01-01", end="2025-12-01", freq="MS")
grid = pd.MultiIndex.from_product([all_items, all_months], names=['Item_ID', 'Date']).to_frame().reset_index(drop=True)

# Merge raw data into grid
df_complete = pd.merge(grid, df_raw, on=['Item_ID', 'Date'], how='left')

# Impute metadata and price rates using forward-fill then backward-fill per item
metadata_cols = ['Item_Name', 'Group', 'Category', 'GP_Index_No', 'pluno', 'W_Rate', 'R_Rate', 'Margin_Abs', 'Margin_Pct']
df_complete[metadata_cols] = df_complete.groupby('Item_ID')[metadata_cols].ffill()
df_complete[metadata_cols] = df_complete.groupby('Item_ID')[metadata_cols].bfill()

# Recompute time-based index fields
df_complete['Month'] = df_complete['Date'].dt.month
df_complete['Year'] = df_complete['Date'].dt.year
df_complete['Quarter'] = df_complete['Date'].dt.quarter
date_to_idx = {d: i+1 for i, d in enumerate(all_months)}
df_complete['Time_Index'] = df_complete['Date'].map(date_to_idx)

# Find first actual sale date of each item to prevent back-filling sales before an item existed
first_sale_df = df_raw.dropna(subset=['Net_Qty']).groupby('Item_ID')['Date'].min().reset_index().rename(columns={'Date': 'First_Date'})
df_complete = pd.merge(df_complete, first_sale_df, on='Item_ID', how='left')

# Impute Net_Qty using linear interpolation per item
df_complete['Net_Qty'] = df_complete.groupby('Item_ID')['Net_Qty'].transform(
    lambda x: x.interpolate(method='linear')
)

# If Date is before First_Date, set Net_Qty to 0.0 (prevents pre-introduction leakage)
df_complete.loc[df_complete['Date'] < df_complete['First_Date'], 'Net_Qty'] = 0.0
df_complete['Net_Qty'] = df_complete['Net_Qty'].fillna(0.0)
df_complete = df_complete.drop(columns=['First_Date'])

# Recompute Lag features and Rolling Mean 3M on the completed timeline
df_complete = df_complete.sort_values(by=['Item_ID', 'Date']).reset_index(drop=True)
df_complete['Lag_1'] = df_complete.groupby('Item_ID')['Net_Qty'].shift(1)
df_complete['Lag_2'] = df_complete.groupby('Item_ID')['Net_Qty'].shift(2)
df_complete['Lag_3'] = df_complete.groupby('Item_ID')['Net_Qty'].shift(3)
df_complete['Rolling_Mean_3M'] = df_complete.groupby('Item_ID')['Net_Qty'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)
df_complete[['Lag_1', 'Lag_2', 'Lag_3', 'Rolling_Mean_3M']] = df_complete[['Lag_1', 'Lag_2', 'Lag_3', 'Rolling_Mean_3M']].fillna(0.0)

print(f"Completed and imputed dataset shape: {df_complete.shape}")
print("Missing values after imputation:")
print(df_complete.isnull().sum())

### **Visualizing Imputation**

Let's visualize the completed sales timeline for a sample product that suffered from missing raw data files.

In [ ]:
# Plot comparison for a sample item
sample_item = "GELLETE PRESTO 5S R/SET"
raw_item = df_raw[df_raw['Item_Name'] == sample_item].sort_values('Date')
imputed_item = df_complete[df_complete['Item_Name'] == sample_item].sort_values('Date')

plt.figure(figsize=(14, 6))
plt.plot(imputed_item['Date'], imputed_item['Net_Qty'], label='Completed (Imputed) Sales', marker='o', linestyle='-', color='green', linewidth=2)
plt.scatter(raw_item['Date'], raw_item['Net_Qty'], label='Raw (Original) Data Points', color='red', s=100, zorder=5)
plt.title(f"Data Completion & Imputation for: {sample_item}", fontsize=14)
plt.xlabel("Date")
plt.ylabel("Net Units Sold")
plt.legend()
plt.show()

## **4. Model Comparison (ARIMA vs. SARIMA vs. XGBoost)**

We evaluate ARIMA, SARIMA, and XGBoost on the top selling products. We use the final 3 months of 2025 (Oct, Nov, Dec) as the validation set, and measure performance using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and Mean Absolute Percentage Error (MAPE).

In [ ]:
# Identify the top 5 highest-selling items by total volume
top_items = df_complete.groupby('Item_Name')['Net_Qty'].sum().sort_values(ascending=False).head(5).index.tolist()
print(f"Comparing models on the top highest volume items: {top_items}\n")

comparison_results = []

for item_name in top_items:
    item_df = df_complete[df_complete['Item_Name'] == item_name].sort_values('Date').reset_index(drop=True)
    
    # Train-test split (Train: 21 months, Validate: 3 months - Oct, Nov, Dec 2025)
    train_series = item_df.iloc[:-3]['Net_Qty'].reset_index(drop=True)
    val_series = item_df.iloc[-3:]['Net_Qty'].reset_index(drop=True)
    
    # 1. ARIMA(1, 1, 1) model
    try:
        model_arima = ARIMA(train_series, order=(1, 1, 1))
        res_arima = model_arima.fit()
        preds_arima = np.clip(res_arima.forecast(steps=3).values, 0, None)
    except Exception as e:
        preds_arima = np.array([train_series.iloc[-1]] * 3) # fallback to naive last value
        
    # 2. SARIMA(1, 1, 1)x(1, 1, 0, 12) seasonal model
    try:
        # Since seasonal cycle is 12, we fit seasonal model
        model_sarima = SARIMAX(train_series, order=(1, 1, 1), seasonal_order=(1, 1, 0, 12), enforce_stationarity=False)
        res_sarima = model_sarima.fit(disp=False)
        preds_sarima = np.clip(res_sarima.forecast(steps=3).values, 0, None)
    except Exception as e:
        preds_sarima = np.array([train_series.iloc[-1]] * 3) # fallback
        
    # 3. Local XGBoost model (trained on this item's lags)
    try:
        features = ['Lag_1', 'Lag_2', 'Lag_3', 'Rolling_Mean_3M', 'Month', 'Quarter']
        X_train = item_df.iloc[:-3][features]
        y_train = item_df.iloc[:-3]['Net_Qty']
        
        # Fit simple XGBRegressor
        model_xgb = xgb.XGBRegressor(n_estimators=50, max_depth=3, learning_rate=0.1, random_state=42)
        model_xgb.fit(X_train, y_train)
        
        # Predict on validation set
        X_val = item_df.iloc[-3:][features]
        preds_xgb = np.clip(model_xgb.predict(X_val), 0, None)
    except Exception as e:
        preds_xgb = np.array([train_series.iloc[-1]] * 3) # fallback
        
    # Evaluate MAE and RMSE
    def eval_errors(preds):
        mae = mean_absolute_error(val_series, preds)
        rmse = np.sqrt(mean_squared_error(val_series, preds))
        mape = np.mean(np.abs((val_series.values - preds) / (val_series.values + 1e-5))) * 100
        return mae, rmse, mape

    mae_a, rmse_a, mape_a = eval_errors(preds_arima)
    mae_s, rmse_s, mape_s = eval_errors(preds_sarima)
    mae_x, rmse_x, mape_x = eval_errors(preds_xgb)
    
    comparison_results.append({
        'Item_Name': item_name,
        'ARIMA_MAE': mae_a, 'ARIMA_RMSE': rmse_a, 'ARIMA_MAPE': mape_a,
        'SARIMA_MAE': mae_s, 'SARIMA_RMSE': rmse_s, 'SARIMA_MAPE': mape_s,
        'XGBoost_MAE': mae_x, 'XGBoost_RMSE': rmse_x, 'XGBoost_MAPE': mape_x
    })

# Convert comparison to DataFrame and show average performance
comp_df = pd.DataFrame(comparison_results)
avg_metrics = pd.DataFrame({
    'Model': ['ARIMA', 'SARIMA', 'XGBoost'],
    'Mean MAE': [comp_df['ARIMA_MAE'].mean(), comp_df['SARIMA_MAE'].mean(), comp_df['XGBoost_MAE'].mean()],
    'Mean RMSE': [comp_df['ARIMA_RMSE'].mean(), comp_df['SARIMA_RMSE'].mean(), comp_df['XGBoost_RMSE'].mean()],
    'Mean MAPE (%)': [comp_df['ARIMA_MAPE'].mean(), comp_df['SARIMA_MAPE'].mean(), comp_df['XGBoost_MAPE'].mean()]
})
print("Average Validation Performance across top 5 items:")
print(avg_metrics.round(2))

In [ ]:
# Plot comparison for the highest volume item
vis_item = top_items[0]
item_df = df_complete[df_complete['Item_Name'] == vis_item].sort_values('Date').reset_index(drop=True)
val_dates = item_df.iloc[-3:]['Date']
val_actuals = item_df.iloc[-3:]['Net_Qty']

# Fit and forecast again for plotting
model_arima = ARIMA(item_df.iloc[:-3]['Net_Qty'], order=(1,1,1)).fit()
model_sarima = SARIMAX(item_df.iloc[:-3]['Net_Qty'], order=(1,1,1), seasonal_order=(1,1,0,12), enforce_stationarity=False).fit(disp=False)

features = ['Lag_1', 'Lag_2', 'Lag_3', 'Rolling_Mean_3M', 'Month', 'Quarter']
model_xgb = xgb.XGBRegressor(n_estimators=50, max_depth=3, learning_rate=0.1, random_state=42)
model_xgb.fit(item_df.iloc[:-3][features], item_df.iloc[:-3]['Net_Qty'])
preds_xgb = model_xgb.predict(item_df.iloc[-3:][features])

plt.figure(figsize=(14, 6))
# Historical training plot
plt.plot(item_df.iloc[:-3]['Date'], item_df.iloc[:-3]['Net_Qty'], label='Historical Training Sales', color='blue', marker='o')
# Actual validation plot
plt.plot(val_dates, val_actuals, label='Actual Sales (Validation)', color='black', marker='x', linewidth=2)
# Predictions
plt.plot(val_dates, np.clip(model_arima.forecast(steps=3), 0, None), label='ARIMA Forecast', linestyle='--', marker='^', color='orange')
plt.plot(val_dates, np.clip(model_sarima.forecast(steps=3), 0, None), label='SARIMA Forecast', linestyle='--', marker='s', color='green')
plt.plot(val_dates, np.clip(preds_xgb, 0, None), label='XGBoost Forecast', linestyle='--', marker='d', color='red')

plt.title(f"Model Forecasting Comparison for: {vis_item}", fontsize=14)
plt.xlabel("Date")
plt.ylabel("Net Units Sold")
plt.legend()
plt.show()

## **5. Full Dataset Preprocessing & Feature Encoding**

We align categoricals (`Group` and `Category`) using One-Hot Encoding and reindex columns to match the 18 production features of the XGBoost Demand Model.

In [ ]:
MODEL_FEATURES = [
    'W_Rate', 'R_Rate', 'Margin_Abs', 'Margin_Pct',
    'Month', 'Year', 'Quarter', 'Time_Index',
    'Lag_1', 'Lag_2', 'Lag_3', 'Rolling_Mean_3M',
    'Group_II', 'Group_III', 'Group_IV', 'Group_V', 'Group_VI',
    'Category_Liquor'
]

cols_to_drop = [
    'Item_ID', 'Date', 'GP_Index_No', 'pluno', 'Item_Name', 
    'Qty', 'Refund_Qty', 'R_Amt', 'W_Amt', 'Profit', 
    'O_B', 'Closing_Stock', 'Net_Tax', 'Net_Qty'
]

# We will use the completed 2024-2025 dataset to train our final production XGBoost model
y_full = df_complete['Net_Qty'].fillna(0.0)
X_full = df_complete.drop(columns=cols_to_drop, errors='ignore').copy()

# One-hot encode categoricals Group and Category
X_full = pd.get_dummies(X_full, columns=['Group', 'Category'], drop_first=True)

# Align columns with exact expected MODEL_FEATURES
X_full = X_full.reindex(columns=MODEL_FEATURES, fill_value=0)

print(f"X_full shape: {X_full.shape}")
print(f"y_full shape: {y_full.shape}")

## **6. Final Production Model Training**

Train the final `XGBRegressor` on 100% of the completed and cleaned 2024-2025 canteen records.

In [ ]:
# Instantiate final XGBoost Regressor matching backend production hyperparameters
final_model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)

print("Training final production model on the completed 2024-2025 dataset...")
final_model.fit(X_full, y_full)
print("Final production model training completed!")

## **7. Save Selected Model**

Save the serialized XGBoost JSON representation to `model/xgboost_demand_model.json` to deploy it to production.

In [ ]:
model_dir = Path("model")
model_dir.mkdir(exist_ok=True)
model_path = model_dir / "xgboost_demand_model.json"

# Save the trained model
final_model.save_model(str(model_path))
print(f"Production model successfully saved to: {model_path.resolve()}")

## **8. Future Recursive Forecasting Simulation (Jan 2026)**

Demonstrate generating recursive forecasts into the future starting from December 2025 state.

In [ ]:
# Simulate predicting demand for January 2026 from Dec 2025 state
print("Simulating future recursive forecasting for Jan 2026...")

# Get the latest state (December 2025)
last_date = df_complete['Date'].max()
last_state = df_complete[df_complete['Date'] == last_date].copy()

# Advance lags by one step
future_state = last_state.copy()
future_state['Lag_3'] = future_state['Lag_2']
future_state['Lag_2'] = future_state['Lag_1']
future_state['Lag_1'] = future_state['Net_Qty'].fillna(0)
future_state['Rolling_Mean_3M'] = (future_state['Lag_1'] + future_state['Lag_2'] + future_state['Lag_3']) / 3

# Compute January 2026 time indexes
next_date = last_date + pd.DateOffset(months=1)
future_state['Month'] = next_date.month
future_state['Year'] = next_date.year
future_state['Quarter'] = (next_date.month - 1) // 3 + 1
future_state['Time_Index'] = int(df_complete['Time_Index'].max()) + 1

# Preprocess features
X_future = future_state.drop(columns=cols_to_drop, errors='ignore').copy()
X_future = pd.get_dummies(X_future, columns=['Group', 'Category'], drop_first=True)
X_future = X_future.reindex(columns=MODEL_FEATURES, fill_value=0)

# Predict demand
future_preds = np.clip(final_model.predict(X_future), 0, None)
future_state['Predicted_Demand'] = np.round(future_preds, 1)

print("\nSample predictions for January 2026 (first 10 items):")
print(future_state[['Item_Name', 'Group', 'Category', 'Closing_Stock', 'Predicted_Demand']].head(10))